# 02 - Selection versus generation, on one encoder

The comparison that isolates the claim: one encoder, one training loop, one seed,
one dataset, and **only the output interface differs**. `span` emits two indices;
`generative` emits characters over a closed 70-character alphabet.

This notebook trains both at a reduced scale so it runs in a few minutes. The
shipped numbers come from `scripts/run_experiments.py` at full scale; the tables
read from `results/tables/` at the bottom are those.

In [ ]:
import sys, os
os.environ.setdefault("OMP_NUM_THREADS", "2")
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import torch; torch.set_num_threads(2)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
print("torch", torch.__version__)


In [ ]:
from gdx.config import load_config
from gdx.pipelines.core import prepare, train_head
from gdx.arms import ARM_BY_NAME, model_candidates
from gdx.pipelines.core import evaluate_arm

cfg = load_config("../configs/base.yaml", ["data.n_train=400", "data.n_val=100", "data.n_test=150", "optim.epochs=4"])
prepared = prepare(cfg, seed=0)
print(prepared.splits.sizes)

In [ ]:
models, candidates = {}, {}
for head in ("span", "generative"):
    models[head], hist = train_head(cfg, prepared, head)
    candidates[head] = model_candidates(models[head], prepared.splits.test, cfg)
    print(head, "params", hist["n_params"], "train seconds", round(hist["train_seconds"], 1))

In [ ]:
rows = []
for arm_name in ("generative", "generative_verify", "span_only", "span_verify", "span_verify_norm"):
    arm = ARM_BY_NAME[arm_name]
    result = evaluate_arm(arm, prepared.splits.test, candidates[arm.source], cfg)
    rows.append({
        "arm": arm_name,
        "strict": result.summary["strict_accuracy"],
        "canonical": result.summary["canonical_accuracy"],
        "coverage": result.summary["coverage"],
        "hallucination": result.summary["hallucination_rate"],
        "grounding_exact": result.summary["grounding_exact"],
    })
pd.DataFrame(rows).round(4)

The `hallucination` column is the point. For the span arms it is exactly 0 and
cannot be otherwise. For `generative` it is whatever the decoder produced.

`generative_verify` is the arm a fair reading demands: the *same* verification
loop applied to generated strings. If it also reaches 0, then the check rather
than the head is what removes hallucinated values -- and that is reported rather
than avoided.

In [ ]:
# What the generative head actually emits, next to the truth.
gen = models["generative"]
batch = prepared.splits.test.collate(list(range(6)))
preds = gen.predict(batch)
rows = []
for doc, per_doc in zip(batch.docs, preds):
    for pred in per_doc:
        truth = doc.fields[pred.field_name]
        rows.append({
            "doc": doc.doc_id, "field": pred.field_name,
            "generated": pred.text, "target": truth.value,
            "in_document": doc.contains_value(pred.field_name, pred.text) if pred.text else None,
        })
pd.DataFrame(rows).head(24)

### The shipped tables

Everything below is read from `results/tables/`, produced by
`scripts/run_experiments.py` and `scripts/analyse.py` at full scale over three
seeds.

In [ ]:
import pathlib
T = pathlib.Path("../results/tables")
pd.read_csv(T / "method_comparison.csv")

In [ ]:
pd.read_csv(T / "verdicts.csv")

In [ ]:
pd.read_csv(T / "statistical_tests.csv")